Process for creating an experiment:

Things we input:
- How many steps each simulation should have.
- Perhaps a trader in question - could start by just graphing their wealth, or checking their wealth at the end.
    - if its specifically for one trader and we want the results to be average wealth accumulated or rate of wealth gain - wpould just be wealth at end - start / time steps or could do sharpe ratio
    - we can then define the trader and their strategies

- 

ideally we want to change to something where we can add in the starting levels, maybe we do an apply level function copying the live data version.

In [ ]:
from dataclasses import dataclass

from traders import Trader

@dataclass
class experiment:
    target: Trader
    independent_variable: str
    dependent_variable: str


    num_steps: int
    decision_seeds: list[int]
    parameter_seeds: list[int]
    

we have our target trader.

1. fix first independent variable value from list, 
2. then run sim on each seed for num_steps. 
3. Record targets - ie append to a list things like avg pnl per step, sharpe ratio, max drawdown if thats appropriate.
4. run over each seed then average the values.
5. move to next value for independent variable.
6. produce graphs of the average for each of the items in question!

In [ ]:
import sys
sys.path.insert(0, "..")

from simulation import Simulation
from strategies import Strategy, random_strat, inventorymm, naivemm

from traders import Trader
import json
import random
from dataclasses import dataclass


'''config = {
    "random_trader" : {
            qty: 80,
            trader_cfg: {},
            strategy_cfg: {},
        },
    "naive_market_maker": {
            qty: 20,
            trader_cfg: {},
            strategy_cfg: {},
        },
    }'''


@dataclass
class TraderConfig:
    min_starting_position: int = 100
    max_starting_position: int = 100
    min_starting_cash: int = 10000
    max_starting_cash: int = 10000
    min_latency: int = 1
    max_latency: int = 100
    min_latency_dev: int = 0
    max_latency_dev: int = 20

@dataclass
class RandomConfig:
    min_buy_prob: float
    max_buy_prob: float
    min_sell_prob: float
    max_sell_prob: float
    min_cancel_prob: float
    max_cancel_prob: float
    min_limit_prob: float
    max_limit_prob: float
    min_max_quantity: int
    max_max_quantity: int
    min_max_price_offest: int
    max_max_price_offset: int
    reference_price: int

@dataclass
class NaiveMMConfig:
    min_half_spread: int
    max_half_spread: int
    min_time_req: int
    max_time_req: int
    min_tolerance: int
    max_tolerance: int
    min_quantity: int
    max_quantity: int


@dataclass
class InventoryMMConfig:
    min_half_spread: int
    max_half_spread: int
    min_time_req: int
    max_time_req: int
    min_tolerance: int
    max_tolerance: int
    min_quantity: int
    max_quantity: int
    min_target_ratio: float
    max_target_ratio: float
    min_ratio_error: float
    max_ratio_error: float


class ConfigParser:
    def __init__(self):
        self.map = {
            "random_trader": self.parse_random_trader,
            "naive_mm": self.parse_naive_mm,
            "inventory_aware_mm": self.parse_inventory_mm,
        }

    def parse_base_trader(self, config: dict) -> TraderConfig:
        try:
            return TraderConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid trader configuration: {e}") from e
        

    def parse_random_trader(self, config: dict) -> RandomConfig:
        try:
            return RandomConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid strategy configuration: {e}") from e
        

    def parse_naive_mm(self, config: dict):
        try:
            return NaiveMMConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid strategy configuration: {e}") from e

        
    def parse_inventory_mm(self, config: dict):
        try:
            return InventoryMMConfig(**config)
        except TypeError as e:
            raise ValueError(f"Invalid strategy configuration: {e}") from e


class TraderFactory:
    def __init__(self, seed):
        self.rng = random.Random(seed)
        self.map = {
            "random_trader": self.create_random,
            "naive_mm": self.create_naive_mm,
            "inventory_aware_mm": self.create_inventory_mm,
        }

    def create_trader(self, data: tuple[str, TraderConfig, RandomConfig | NaiveMMConfig | InventoryMMConfig], id: int, sim: Simulation) -> Trader:
        # unpack data
        strat = self.map[data[0]]
        trader_cfg = data[1]
        strat_cfg = data[2]

        return Trader(model=sim,
                      trader_id=id,
                      initial_cash=self.rng.randint(trader_cfg.min_starting_cash, trader_cfg.max_starting_cash),
                      initial_position=self.rng.randint(trader_cfg.min_starting_position, trader_cfg.max_starting_position),
                      latency=self.rng.randint(trader_cfg.min_latency, trader_cfg.max_latency),
                      latency_deviation=self.rng.randint(trader_cfg.min_latency_dev, trader_cfg.max_latency_dev),
                      strategy=strat(strat_cfg),
                      )


    def create_random(self, data: RandomConfig) -> Strategy:
        min_total = data.min_buy_prob + data.min_sell_prob
        both_fixed = (data.min_buy_prob == data.max_buy_prob 
                      and data.min_sell_prob == data.max_sell_prob)
        

        if min_total > 1 or (min_total == 1 and not both_fixed):
            raise ValueError(
                "Buy/sell ranges must allow sampling probabilities "
                "whose sum is at most 1."
            )

        buy_prob = self.rng.uniform(data.min_buy_prob, data.max_buy_prob)
        sell_prob = self.rng.uniform(data.min_sell_prob, data.max_sell_prob)

        while buy_prob + sell_prob > 1:
            buy_prob = self.rng.uniform(data.min_buy_prob, data.max_buy_prob)
            sell_prob = self.rng.uniform(data.min_sell_prob, data.max_sell_prob)

        return random_strat.Random(
            buy_probability=buy_prob,
            sell_probability=sell_prob,
            cancel_probability=self.rng.uniform(data.min_cancel_prob, data.max_cancel_prob),
            limit_probability=self.rng.uniform(data.min_limit_prob, data.max_limit_prob),
            max_quantity=self.rng.randint(data.min_max_quantity, data.max_max_quantity),
            max_price_offset=self.rng.randint(data.min_max_price_offest, data.max_max_price_offset),
            reference_price=data.reference_price,
        )



    def create_naive_mm(self, data: NaiveMMConfig) -> Strategy:
        return naivemm.NaiveMM(
            half_spread=self.rng.randint(data.min_half_spread, data.max_half_spread),
            time_req=self.rng.randint(data.min_time_req, data.max_time_req),
            tolerance=self.rng.randint(data.min_tolerance, data.max_tolerance),
            quantity=self.rng.randint(data.min_quantity, data.max_quantity),
        )

    def create_inventory_mm(self, data: InventoryMMConfig) -> Strategy:
        return inventorymm.InventoryMM(
            half_spread=self.rng.randint(data.min_half_spread, data.max_half_spread),
            time_req=self.rng.randint(data.min_time_req, data.max_time_req),
            tolerance=self.rng.randint(data.min_tolerance, data.max_tolerance),
            target_ratio=self.rng.uniform(data.min_target_ratio, data.max_target_ratio),
            ratio_error=self.rng.uniform(data.min_ratio_error, data.max_ratio_error),
            quantity=self.rng.randint(data.min_quantity, data.max_quantity),
        )


def factory(trader_config: str, decision_seed, trader_parameter_seed) -> Simulation:
    sim: Simulation = Simulation(seed=decision_seed)
    parser: ConfigParser = ConfigParser()
    trader_factory: TraderFactory = TraderFactory(seed=trader_parameter_seed)

    trader_id = 1


    config = json.loads(trader_config)

    # create the config classes, get their quantities:
    config_classes = []


    for strategy in config:
        try:
            parse_function = parser.map[strategy]
        except KeyError:
            raise ValueError(f"Unknown trader strategy: {strategy}")

        qty = config[strategy]["quantity"]
        trader_cfg = parser.parse_base_trader(config[strategy]["trader_cfg"])
        strategy_cfg = parse_function(config[strategy]["strategy_cfg"])

        config_classes.append(((strategy, trader_cfg, strategy_cfg), qty))

    # config_classes is now a list of tuples: [(strategy_name, quantity, trader_config, strategy_config)]
    # want to process each 

    for config, qty in config_classes:
        for i in range(qty):
            # pass in config to create trader:
            trader: Trader = trader_factory.create_trader(config, trader_id, sim=sim)
            sim.traders[trader_id] = trader
            trader_id += 1


    return sim







In [21]:
trader_config = '''{
  "random_trader": {
    "quantity": 20,
    "trader_cfg": {},
    "strategy_cfg": {
      "min_buy_prob": 0.2,
      "max_buy_prob": 0.6,
      "min_sell_prob": 0.2,
      "max_sell_prob": 0.6,
      "min_cancel_prob": 0.01,
      "max_cancel_prob": 0.1,
      "min_limit_prob": 0.5,
      "max_limit_prob": 1.0,
      "min_max_quantity": 1,
      "max_max_quantity": 5,
      "min_max_price_offest": 0,
      "max_max_price_offset": 3,
      "reference_price": 100
    }
  },
  "naive_mm": {
    "quantity": 5,
    "trader_cfg": {},
    "strategy_cfg": {
      "min_half_spread": 1,
      "max_half_spread": 2,
      "min_time_req": 1,
      "max_time_req": 5,
      "min_tolerance": 0,
      "max_tolerance": 2,
      "min_quantity": 1,
      "max_quantity": 5
    }
  },
  "inventory_aware_mm": {
    "quantity": 5,
    "trader_cfg": {},
    "strategy_cfg": {
      "min_half_spread": 1,
      "max_half_spread": 2,
      "min_time_req": 1,
      "max_time_req": 5,
      "min_tolerance": 0,
      "max_tolerance": 2,
      "min_quantity": 1,
      "max_quantity": 5,
      "min_target_ratio": 0.4,
      "max_target_ratio": 0.6,
      "min_ratio_error": 0.05,
      "max_ratio_error": 0.1
    }
  }
}'''

In [22]:
sim = factory(trader_config=trader_config,
              decision_seed=42,
              trader_parameter_seed=123)

In [32]:
sim.run_sim(10000)

In [ ]:
for trader_id, trader in sim.traders.items():
    print(
        f"Trader {trader_id}: "
        f"strategy={type(trader.strategy).__name__}, "
        f"cash={trader.current_cash}, "
        f"position={trader.current_position}"
    )

Need a function to create a trader from our specific parameters should fix everything apart from one item. then should take this item from a list. 

This is the variable on the outside, so we first need to measure the randomness for each variable and then do the outside change later

so we need something that creates the initial conditions for the trader. we can just use the parsing functions but with min/max the same?

In [ ]:
from __future__ import annotations
import sys
sys.path.insert(0, "..")

from traders import Trader
from factory import factory
from collections import defaultdict
from dataclasses import dataclass
from simulation import Simulation
from strategies import Strategy, random_strat, inventorymm, naivemm
from enum import Enum
import statistics
from models import TraderResults


class IndependentVariableScope(Enum):
    TRADER = "trader"
    STRATEGY = "strategy"


'''target_cfg = {
    "strategy_name": "naive_mm",
    "trader_cfg": {
        "starting_position": 100,
        "starting_cash": 10000,
        "latency": None,
        "latency_dev": 5,
    },
    "strategy_cfg": {
        "half_spread": 2,
        "time_req": 5,
        "tolerance": 1,
        "quantity": 3,
    }
}'''

@dataclass
class TargetRandomConfig:
    buy_probability: float | None
    sell_probability: float | None
    cancel_probability: float | None
    limit_probability: float | None
    max_quantity: int | None
    max_price_offset: int | None
    reference_price: int | None


@dataclass
class TargetNaiveMMConfig:
    half_spread: int | None
    time_req: int | None
    tolerance: int | None
    quantity: int | None


@dataclass
class TargetInventoryMMConfig:
    half_spread: int | None
    time_req: int | None
    tolerance: int | None
    target_ratio: float | None
    ratio_error: float | None
    quantity: int | None

@dataclass
class TargetTraderConfig:
    initial_cash: int | None
    initial_position:int | None
    latency: int | None
    latency_deviation: int | None
    strategy: TargetRandomConfig | TargetNaiveMMConfig | TargetInventoryMMConfig
    trader_id: int = 0



class TargetTraderFactory:
    def __init__(self):
        self.strategy_builders = {
            TargetRandomConfig: self.create_random,
            TargetNaiveMMConfig: self.create_naive_mm,
            TargetInventoryMMConfig: self.create_inventory_mm,
        }

    def create_trader(self, cfg: TargetTraderConfig, sim: Simulation) -> Trader:
        strategy = self.create_strategy(cfg.strategy)

        return Trader(
            model=sim,
            trader_id=cfg.trader_id,
            initial_cash=cfg.initial_cash,
            initial_position=cfg.initial_position,
            latency=cfg.latency,
            latency_deviation=cfg.latency_deviation,
            strategy=strategy,
        )

    def create_strategy(self, cfg):
        try:
            builder = self.strategy_builders[type(cfg)]
        except KeyError:
            raise TypeError(f"Unsupported target strategy config: {type(cfg)}")

        return builder(cfg)

    def create_random(self, cfg: TargetRandomConfig):
        return random_strat.Random(
            buy_probability=cfg.buy_probability,
            sell_probability=cfg.sell_probability,
            cancel_probability=cfg.cancel_probability,
            limit_probability=cfg.limit_probability,
            max_quantity=cfg.max_quantity,
            max_price_offset=cfg.max_price_offset,
            reference_price=cfg.reference_price,
        )

    def create_naive_mm(self, cfg: TargetNaiveMMConfig):
        return naivemm.NaiveMM(
            half_spread=cfg.half_spread,
            time_req=cfg.time_req,
            tolerance=cfg.tolerance,
            quantity=cfg.quantity,
        )

    def create_inventory_mm(self, cfg: TargetInventoryMMConfig):
        return inventorymm.InventoryMM(
            half_spread=cfg.half_spread,
            time_req=cfg.time_req,
            tolerance=cfg.tolerance,
            target_ratio=cfg.target_ratio,
            ratio_error=cfg.ratio_error,
            quantity=cfg.quantity,
        )



class ExperimentTrader:
    def __init__(self, ind_var: str,
                 ind_var_values: list,
                 target_trader_parameters,
                 sim_config,
                 decision_seeds: list[int],
                 trader_parameter_seeds: list[int],
                 sim_steps: int,
                 ind_var_scope: IndependentVariableScope):
        self.ind_var = ind_var
        self.ind_var_values = ind_var_values
        self.target_trader_parameters = target_trader_parameters
        self.sim_config = sim_config
        self.decision_seeds = decision_seeds
        self.trader_parameter_seeds = trader_parameter_seeds
        self.sim_steps = sim_steps
        self.ind_var_scope = ind_var_scope


    def run_experiment(self):
        main_results = {} # maybe something like {value: {mean: , variance: , std_dev: }} is this better than just [val, mean, var, std]
        target_cfg = self.initialise_target_trader_config()

        variable_cfg = (target_cfg.strategy if self.ind_var_scope 
                        is IndependentVariableScope.STRATEGY
                        else target_cfg)

        if not hasattr(variable_cfg, self.ind_var):
            raise ValueError(
                f"{self.ind_var!r} is not a valid parameter for "
                f"{type(variable_cfg).__name__}"
            )
        
        
        for value in self.ind_var_values:
            # finish the target_cfg
            setattr(variable_cfg, self.ind_var, value)
            results_for_independent_value: list[TraderResults] = []   # each value should finish length n*m for number of seed combinations

            for decision_seed in self.decision_seeds:
                for parameter_seed in self.trader_parameter_seeds:
                    # set up simulation, create trader and attach
                    sim = factory(self.sim_config, decision_seed=decision_seed, trader_parameter_seed=parameter_seed)
                    target_trader = self.create_target_trader(target_cfg, sim)
                    sim.traders[0] = target_trader

                    # run simulation for N steps
                    sim.run_sim(self.sim_steps)
                    results_for_seed: TraderResults = sim.results_for_trader(trader_id = target_trader.trader_id)
                    results_for_independent_value.append(results_for_seed)


            res = self.get_statistics(results_for_independent_value)
            main_results[value] = res


        return main_results




    def get_statistics(self, results: list[TraderResults]) -> dict:
        summary = {}
        
        for metric in ("total_pnl", "avg_pnl", "sharpe_ratio", "drawdown"):
            values = [getattr(result, metric) for result in results]
            values = [value for value in values if value is not None]

            summary[metric] = {
                "count": len(values),
                "mean": statistics.mean(values) if values else None,
                "variance": statistics.variance(values) if len(values) >= 2 else None,
            }

        return summary



    def initialise_target_trader_config(self) -> TargetTraderConfig:
        strategy_config_map = {
            "random_trader": TargetRandomConfig,
            "naive_mm": TargetNaiveMMConfig,
            "inventory_aware_mm": TargetInventoryMMConfig,
        }

        params = self.target_trader_parameters
        strategy_cfg = strategy_config_map[params["strategy_name"]](
            **params["strategy_cfg"]
        )

        return TargetTraderConfig(
            **params["trader_cfg"],
            strategy=strategy_cfg,
        )

    def create_target_trader(self, cfg, sim) -> Trader:
        trader_factory = TargetTraderFactory()
        return trader_factory.create_trader(cfg, sim)



